## Processing Messages from Redis Queue (Subscriber)

### Installing Utilities and Libraries

In [ ]:
%pip install python-dotenv redis==8.0.0 openai==2.38.0

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# loading redis configurations
redis_hostname = os.getenv("REDIS_HOSTNAME")
redis_password = os.getenv("REDIS_PASSWORD")

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Setting up the Redis Client

In [ ]:
from redis.cluster import RedisCluster
import ssl

r = RedisCluster(
    host=redis_hostname,
    port=10000,
    password=redis_password,
    ssl=True,
    ssl_cert_reqs=ssl.CERT_NONE,
    ssl_check_hostname=False,
    decode_responses=True
)

### Create the Chat Completions Function

In [ ]:
from openai import AzureOpenAI 

def process_user_message(user_query):

    # creating the Azure OpenAI client to process user requests
    azure_openai_client = AzureOpenAI(
        api_key=azure_openai_api_key,
        api_version="2024-02-15-preview",
        azure_endpoint=azure_openai_endpoint
    )
    
    # sending a chat completions request to the LLM
    chat_completions_response = azure_openai_client.chat.completions.create(
        model = chat_completions_model_name,
        messages = [
            {"role": "system", "content": "You are a helpful AI assistant"},
            {"role": "user", "content": user_query}
        ],
        temperature=0.7
    )

    return chat_completions_response.choices[0].message.content


### Processing Message from the Queue

In [ ]:
# Worker service processing loop
def process_ai_tasks():
    # One-time setup: create the consumer group
    try:
        r.xgroup_create('ai:inference:queue', 'workers', id='0', mkstream=True)
    except:
        pass # if consumer group already exists then pass
    
    # main processing loop
    while True:
        # Get up to 5 tasks, wait 5 seconds if queue empty
        messages = r.xreadgroup(
            groupname='workers',
            consumername='worker-001',
            streams={'ai:inference:queue': '>'},
            count=5,
            block=5000
        )

        for stream, tasks in messages:
            for task_id, task_data in tasks:
                try:
                    # processing AI request by the user
                    print("processing task with id: {}".format(task_id))
                    print("==========================================")
                    assistant_response = process_user_message(task_data['prompt'])
                    print("Assistant response")
                    print("==================================")
                    print(assistant_response)

                    # mark task as complete
                    r.xack('ai:inference:queue', 'workers', task_id)
                except Exception as e:
                    # Task stays in pending, will retry
                    print("Encountered error: {}".format(e))

In [ ]:
process_ai_tasks()